In [9]:
%pip install qiskit
%pip install qiskit-aer
%pip install pylatexenc
%pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.
/bin/bash: {sys.executable}: command not found


# The 9-qubit Shor code
Now we turn to the 9-qubit Shor code, which is a quantum error correcting code obtained by combining together the two codes considered in the previous section: the 3-bit repetition code for qubits, which allows for the correction of a single bit-flip error, and the modified version of that code, which allows for the correction of a single phase-flip error.

# Code description

The 9-qubit Shor code is the code we obtain by concatenating the two codes from the previous section. This means that we first apply one encoding, which encodes one qubit into three, and then we apply the other encoding to each of the three qubits used for the first encoding, resulting in nine qubits in total.

To be more precise, while we could apply the two codes in either order in this particular case, we'll make the choice to first apply the modified version of the 3-bit repetition code (which detects phase-flip errors), and then we'll encode each of the resulting three qubits independently using the original 3-bit repetition code (which detects bit-flip errors). Here is a circuit diagram representation of this encoding.

In [6]:
from qiskit import QuantumCircuit, QuantumRegister
q = QuantumRegister(9, 'q')
circuit = QuantumCircuit(q)

# First, entangle central qubits with each other
circuit.cx(q[0], q[3])
circuit.cx(q[0], q[6])

# Apply Hadamards 
for center in [0, 3, 6]:
    circuit.h(center)

# Entangle central qubits with their neighbors
neighbors = [[1, 2], [4, 5], [7, 8]]
for center, nbs in zip([0, 3, 6], neighbors):
    for nb in nbs:
        circuit.cx(center, nb)

#circuit
print(circuit.draw(output='text'))

               ┌───┐          
q_0: ──■────■──┤ H ├──■────■──
       │    │  └───┘┌─┴─┐  │  
q_1: ──┼────┼───────┤ X ├──┼──
       │    │       └───┘┌─┴─┐
q_2: ──┼────┼────────────┤ X ├
     ┌─┴─┐  │  ┌───┐     └───┘
q_3: ┤ X ├──┼──┤ H ├──■────■──
     └───┘  │  └───┘┌─┴─┐  │  
q_4: ───────┼───────┤ X ├──┼──
            │       └───┘┌─┴─┐
q_5: ───────┼────────────┤ X ├
          ┌─┴─┐┌───┐     └───┘
q_6: ─────┤ X ├┤ H ├──■────■──
          └───┘└───┘┌─┴─┐  │  
q_7: ───────────────┤ X ├──┼──
                    └───┘┌─┴─┐
q_8: ────────────────────┤ X ├
                         └───┘


# Correcting phase-flip errors

Next we'll consider phase-flip errors, or Z errors for brevity. This time it's not quite as clear what we should do because the outer code is the one that detects Z errors, but the inner code seems to be somehow "in the way," making the detection and correction of these errors slightly more difficult.

Suppose that a Z error occurs on one of the 9 qubits of the Shor code, such as the one indicated in this diagram.

We've already observed what happens when a \(ZZ\) error occurs when we're using the 3-bit repetition code — it's equivalent to a \(ZZ\) error occurring **prior to encoding**. 

In the context of the 9-qubit Shor code, this means that a \(ZZ\) error on any one of the three qubits within a block always has the same effect, which is equivalent to a \(ZZ\) error occurring on the corresponding qubit **prior to the inner code being applied**.

For example, the circuit diagram above is equivalent to the following diagram. This can be reasoned using the relationships between \(ZZ\) and CNOT gates described above, or by simply evaluating the circuits on an arbitrary qubit state \(|\psi\rangle\).

In [13]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

# 9 qubits for the state, plus ancilla qubits for error detection
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's') # We need syndrome bits to 'read' the error
circuit = QuantumCircuit(q, syndrome)

# 1. ENCODING (Using the "nice" version)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)

circuit.barrier() 
# Phase flip (Z) on the 3rd qubit of the 2nd block (index 5)
circuit.z(0) 
circuit.barrier()

# 3. SYNDROME MEASUREMENT (Simplified Logic)
# To detect a phase flip in the Shor code, we look at the parity between blocks
# To detect a bit flip (which a Z error can look like here), we check parity within blocks
# In block 1, we check if q[3]==q[4] and q[4]==q[5]
# [This is where you'd add ancilla measurements to find the Z error]

print(circuit.draw(output='text'))

               ┌───┐           ░ ┌───┐ ░ 
q_0: ──■────■──┤ H ├──■────■───░─┤ Z ├─░─
       │    │  └───┘┌─┴─┐  │   ░ └───┘ ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [22]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(1) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░ ┌───┐ ░ 
q_1: ──┼────┼───────┤ X ├──┼───░─┤ Z ├─░─
       │    │       └───┘┌─┴─┐ ░ └───┘ ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [15]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(2) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░ ┌───┐ ░ 
q_2: ──┼────┼────────────┤ X ├─░─┤ Z ├─░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░ └───┘ ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [23]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(3) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░ ┌───┐ ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░─┤ Z ├─░─
     └───┘  │  └───┘┌─┴─┐  │   ░ └───┘ ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [24]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(4) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░ ┌───┐ ░ 
q_4: ───────┼───────┤ X ├──┼───░─┤ Z ├─░─
            │       └───┘┌─┴─┐ ░ └───┘ ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [25]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(5) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░ ┌───┐ ░ 
q_5: ───────┼────────────┤ X ├─░─┤ Z ├─░─
          ┌─┴─┐┌───┐     └───┘ ░ └───┘ ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [26]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(6) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░ ┌───┐ ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░─┤ Z ├─░─
          └───┘└───┘┌─┴─┐  │   ░ └───┘ ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░       ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [20]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(7) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░ ┌───┐ ░ 
q_7: ───────────────┤ X ├──┼───░─┤ Z ├─░─
                    └───┘┌─┴─┐ ░ └───┘ ░ 
q_8: ────────────────────┤ X ├─░───────░─
                         └───┘ ░       ░ 
s: 8/════════════════════════════════════
                                         


In [21]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
q = QuantumRegister(9, 'q')
syndrome = ClassicalRegister(8, 's')
circuit = QuantumCircuit(q, syndrome)
for target in [3, 6]:
    circuit.cx(0, target)
for i in [0, 3, 6]:
    circuit.h(i)
for i in [0, 3, 6]:
    circuit.cx(i, i+1)
    circuit.cx(i, i+2)
circuit.barrier() 
circuit.z(8) 
circuit.barrier()
print(circuit.draw(output='text'))

               ┌───┐           ░       ░ 
q_0: ──■────■──┤ H ├──■────■───░───────░─
       │    │  └───┘┌─┴─┐  │   ░       ░ 
q_1: ──┼────┼───────┤ X ├──┼───░───────░─
       │    │       └───┘┌─┴─┐ ░       ░ 
q_2: ──┼────┼────────────┤ X ├─░───────░─
     ┌─┴─┐  │  ┌───┐     └───┘ ░       ░ 
q_3: ┤ X ├──┼──┤ H ├──■────■───░───────░─
     └───┘  │  └───┘┌─┴─┐  │   ░       ░ 
q_4: ───────┼───────┤ X ├──┼───░───────░─
            │       └───┘┌─┴─┐ ░       ░ 
q_5: ───────┼────────────┤ X ├─░───────░─
          ┌─┴─┐┌───┐     └───┘ ░       ░ 
q_6: ─────┤ X ├┤ H ├──■────■───░───────░─
          └───┘└───┘┌─┴─┐  │   ░       ░ 
q_7: ───────────────┤ X ├──┼───░───────░─
                    └───┘┌─┴─┐ ░ ┌───┐ ░ 
q_8: ────────────────────┤ X ├─░─┤ Z ├─░─
                         └───┘ ░ └───┘ ░ 
s: 8/════════════════════════════════════
                                         


This suggests one option for detecting and correcting Z errors, which is to decode the inner code, leaving us with the three qubits used for the outer encoding along with six initialized workspace qubits. We can then check these three qubits of the outer code for Z errors, and then finally we can re-encode using the inner code, to bring us back to the 9-qubit encoding we get from the Shor code. If we do detect a Z error, we can either correct it prior to re-encoding with the inner code, or we can correct it after re-encoding, by applying a Z gate to any of the qubits in that block.

Here's a circuit diagram that includes the encoding circuit and the error suggested above together with the steps just described (but not the actual correction step).